# Week 8 Exercise: Autonomous Deal Negotiation Simulator

## Overview

This exercise builds on the **AutonomousPlanningAgent**, extending the 3-tool agentic loop into a richer **5-tool negotiation workflow**:

| Tool | Purpose |
|------|---------|
| `scan_the_internet_for_bargains` | Scrape RSS feeds for deals
| `estimate_true_value` | Ensemble price estimation
| `find_competing_products` | RAG search in ChromaDB
| `compare_and_decide` | Structured verdict (buy/pass/wait)
| `notify_user_of_deal` | Push notification

The LLM autonomously orchestrates this workflow: scan, estimate, research competitors, compare, and only notify if the verdict is "buy".

# Part 1: Setup

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path

week8_dir = str(Path.cwd().parent.parent)
tobenna_dir = str(Path.cwd())

if week8_dir not in sys.path:
    sys.path.insert(0, week8_dir)
if tobenna_dir not in sys.path:
    sys.path.insert(0, tobenna_dir)

# Agents like NeuralNetworkAgent load weights relative to CWD
os.chdir(week8_dir)

from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
import chromadb
from openai import OpenAI
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

from agents.scanner_agent import ScannerAgent
from agents.deals import Deal, Opportunity

openai = OpenAI()
MODEL = "gpt-5.1"

DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")

print(f"ChromaDB loaded from: {os.path.abspath(DB)}")
print(f"Collection has {collection.count()} products")

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

# Part 2: Concept Walkthrough — Testing Each Tool Individually

We first demonstrate each of the 5 tools independently before combining them into the autonomous agent loop.

### Tool 1: Scan for deals (using test data)

We use `ScannerAgent.test_scan()` to get hardcoded test deals, same as Day 4.

In [ ]:
scanner = ScannerAgent()
test_results = scanner.test_scan()

for deal in test_results.deals:
    print(f"${deal.price:>7.2f} | {deal.product_description[:80]}...")
    print()

### Tool 2: Estimate true value

We use GPT to estimate the price — in the full agent this wraps the EnsembleAgent.
For this walkthrough we call OpenAI directly.

In [ ]:
sample_deal = test_results.deals[0]
print(f"Estimating value for: {sample_deal.product_description[:80]}...")

response = openai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Estimate the price of this product. Respond with just the price, no explanation.\n\n{sample_deal.product_description}"}],
    reasoning_effort="none",
)
estimated_price = response.choices[0].message.content
print(f"Deal price: ${sample_deal.price:.2f}")
print(f"Estimated value: {estimated_price}")

### Tool 3 (NEW): Find competing products via RAG

This is the first new tool. It uses SentenceTransformer + ChromaDB to find similar
products in the vector store.

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

vector = encoder.encode([sample_deal.product_description])
results = collection.query(query_embeddings=vector.astype(float).tolist(), n_results=5)

documents = results["documents"][0][:]
prices = [m["price"] for m in results["metadatas"][0][:]]

print(f"Found {len(documents)} competing products for: {sample_deal.product_description[:60]}...\n")
for doc, price in zip(documents, prices):
    print(f"  ${price:>7.2f} | {doc[:80]}...")
    print()

### Tool 4 (NEW): Compare and decide — Structured Verdict

This is the second new tool. It sends the deal + competitors to OpenAI using
**Structured Outputs** with a `Verdict` Pydantic model to get
a buy/pass/wait decision.

In [ ]:
from negotiation_agent import Verdict

competitors = [{"description": doc, "price": price} for doc, price in zip(documents, prices)]
competing_json = json.dumps(competitors)

user_prompt = (
    f"You are evaluating whether to recommend a deal to a user.\n\n"
    f"Deal description: {sample_deal.product_description}\n"
    f"Deal price: ${sample_deal.price:.2f}\n"
    f"Estimated true value: $300.00\n\n"
    f"Here are similar products from our database for comparison:\n"
    f"{competing_json}\n\n"
    f"Based on the discount (estimated value minus deal price), the competing product prices, "
    f"and the quality of the deal, decide: buy, pass, or wait.\n"
    f"- 'buy' means the deal is compelling and the user should act now.\n"
    f"- 'pass' means the deal is not worth it.\n"
    f"- 'wait' means it might get better or needs more research."
)

response = openai.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": user_prompt}],
    response_format=Verdict,
)

verdict = response.choices[0].message.parsed
print(f"Decision:   {verdict.decision}")
print(f"Confidence: {verdict.confidence:.2f}")
print(f"Reasoning:  {verdict.reasoning}")
print(f"Comparison: {verdict.competing_product_summary}")

### Tool 5: Notify user

This wraps the MessagingAgent — same as the AutonomousPlanningAgent.
We skip actually calling it here (requires Pushover setup), but the full agent handles it.

In [ ]:
print("Tool 5 (notify_user_of_deal) wraps MessagingAgent.notify()")
print("It sends a push notification via Pushover — same as Day 3.")
print("In the full agent, this is only called when the verdict decision is 'buy'.")

# Part 3: The Full Autonomous NegotiationAgent

Now we bring it all together. The `NegotiationAgent` wraps all 5 tools and lets the
LLM autonomously decide the workflow — the same `while not done` agentic loop from Day 4,
but with 5 tools instead of 3.

The LLM must:
1. Scan for deals
2. Estimate the true value of each deal
3. Pick the best deal and find competing products via RAG
4. Compare against competitors and produce a structured verdict
5. Only notify the user if the verdict is "buy"

In [ ]:
from negotiation_agent import NegotiationAgent

agent = NegotiationAgent(collection)

In [ ]:
verdict, opportunity = agent.negotiate(memory=[])

### Results

In [ ]:
if verdict:
    print("=== VERDICT ===")
    print(f"Decision:   {verdict.decision}")
    print(f"Confidence: {verdict.confidence:.2f}")
    print(f"Reasoning:  {verdict.reasoning}")
    print(f"Comparison: {verdict.competing_product_summary}")
    print()

if opportunity:
    print("=== OPPORTUNITY (notification sent) ===")
    print(f"Product:  {opportunity.deal.product_description[:100]}...")
    print(f"Price:    ${opportunity.deal.price:.2f}")
    print(f"Estimate: ${opportunity.estimate:.2f}")
    print(f"Discount: ${opportunity.discount:.2f}")
    print(f"URL:      {opportunity.deal.url}")
else:
    print("No notification sent — verdict was not 'buy' or no deals found.")

# Part 4: Simple Gradio UI

A minimal Gradio interface to display the negotiation results and allow re-running the agent.

In [ ]:
import gradio as gr

def run_negotiation():
    neg_agent = NegotiationAgent(collection)
    v, opp = neg_agent.negotiate(memory=[])

    verdict_text = "No verdict produced."
    if v:
        verdict_text = (
            f"**Decision:** {v.decision.upper()}\n\n"
            f"**Confidence:** {v.confidence:.0%}\n\n"
            f"**Reasoning:** {v.reasoning}\n\n"
            f"**Competitor Analysis:** {v.competing_product_summary}"
        )

    table_data = []
    if opp:
        table_data.append([
            opp.deal.product_description[:120] + "...",
            f"${opp.deal.price:.2f}",
            f"${opp.estimate:.2f}",
            f"${opp.discount:.2f}",
            opp.deal.url,
        ])

    return verdict_text, table_data


with gr.Blocks(title="Deal Negotiation Simulator", fill_width=True) as ui:
    with gr.Row():
        gr.Markdown(
            '<div style="text-align: center; font-size: 24px;">'
            '<strong>Autonomous Deal Negotiation Simulator</strong>'
            '</div>'
        )
    with gr.Row():
        gr.Markdown(
            '<div style="text-align: center; font-size: 14px;">'
            'Autonomous planner with RAG-based competitor search and structured verdicts.'
            '</div>'
        )

    with gr.Row():
        run_btn = gr.Button("Run Negotiation", variant="primary")

    with gr.Row():
        verdict_display = gr.Markdown(label="Verdict", value="*Click 'Run Negotiation' to start...*")

    with gr.Row():
        deals_table = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[5, 1, 1, 1, 3],
            row_count=5,
            col_count=5,
            max_height=300,
        )

    run_btn.click(run_negotiation, outputs=[verdict_display, deals_table])

ui.launch(inbrowser=True)